# Ray Tasks Revisited

© 2019-2022, Anyscale. All Rights Reserved

![Anyscale Academy](../images/AnyscaleAcademyLogo.png)

The [Ray Crash Course](../ray-crash-course/00-Ray-Crash-Course-Overview.ipynb) introduced the core concepts of Ray's API and how they parallelize work. Specifically, we learned how to define Ray _tasks_ and _actors_, run them, and retrieve the results. 

This lesson explores Ray tasks in greater depth, including the following:

* How task dependencies are handled automatically by Ray
* Usage patterns for `ray.get()` and `ray.wait()`
* Specifying limits on the number of invocations and retries on failure
* An exploration of task granularity considerations
* Profiling tasks

In [1]:
import ray, time, os, sys 
import numpy as np 
sys.path.append("..")
from util.printing import pd, pnd  # convenience methods for printing results.

In [2]:
ray.init(ignore_reinit_error=True)

2026-05-06 15:42:10,440	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/Users/ravinkohli/self-attention-from-scratch/academy/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.10
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


The Ray Dashboard URL is printed above and also part of the output dictionary item `webui_url`

(When using the Anyscale platform, use the URL provided by your instructor to access the Ray Dashboard.)

## Ray Task Dependencies

Let's define a few remote tasks, which will have _dependency_ relationships. We'll learn how Ray handles these dependent, asynchronous computations.

One task will return a random NumPy array of some size `n` and the other task will add two such arrays. We'll also add a sleep time, one tenth the size of `n` to simulate expensive computation.

> **Note:** Dependencies and how Ray implements handling of them are explored in depth in the [03: Ray Internals](03-Ray-Internals.ipynb) lesson. 

In [3]:
@ray.remote
def make_array(n):
    time.sleep(n/10.0)
    return np.random.standard_normal(n)

Now define a task that can add two NumPy arrays together. The arrays need to be the same size, but we'll ignore any checking for this requirement.

In [4]:
@ray.remote
def add_arrays(a1, a2):
    time.sleep(a1.size/10.0)
    return np.add(a1, a2)

Now lets use them!

In [5]:
start = time.time()
ref1 = make_array.remote(20)
ref2 = make_array.remote(20)
ref3 = add_arrays.remote(ref1, ref2)
print(ray.get(ref3))
pd(time.time() - start, prefix="Total time:")

[-0.72504476  0.89738743  1.01112603 -0.35459977  0.88724625 -1.30015821
  2.89480641 -1.9008292   2.21737994  0.00400175 -1.42005648 -1.55300039
 -1.31805977  1.82932134  0.11050001 -0.61236042  0.00738695 -1.19447934
  1.16133215 -1.63444642]
Total time: duration:  4.545 seconds


Something subtle and "magical" happened here; when we called `add_arrays`, we didn't need to call `ray.get()` first for `ref1` and `ref2`, since `add_arrays` expects NumPy arrays. Because `add_arrays` is a Ray task, Ray automatically does the extraction for us, so we can write code that looks more natural and Pythonic.

Furthermore, note that the `add_arrays` task effectively depends on the outputs of the two `make_array` tasks. Ray won't run `add_arrays` until the other tasks are finished. Hence, Ray automatically handles task dependencies for us.

This is why the elapsed time is about 4 seconds. We used a size of 20, so we slept 2 seconds in each call to `make_array`, but those happened in parallel, _followed_ by a second sleep of 2 seconds in `add_arrays`.

Even though three task invocations occurred, we only used one call to `ray.get()`, when we actually needed the final results. Eliminating unnecessary `ray.get()` calls helps avoid forcing tasks to become synchronous when they could be asynchronous. So, keep these two key points in mind:

* _Don't ask for results you don't need._
* _Don't ask for the results you need until you really need them._

We don't need to see the objects for `id1` and `id2`. We only need the final array for `id3`.

## Using ray.wait() with ray.get()

Here is an idiomatic way to use `ray.get()`, where we fire all five asynchronous tasks at once, then ask for all the results at once with `ray.get()`:

In [6]:
start = time.time()

# Comprehension list: five NumPy object references or futures created
array_refs = [make_array.remote(n*10) for n in range(5)]

# Comprehension list: object references or futures of the result of addition
added_array_refs = [add_arrays.remote(ref, ref) for ref in array_refs]

# Iterate o er the list of object references or futures
for array in ray.get(added_array_refs):
    print(f'{array.size}: {array}')
pd(time.time() - start, prefix="Total time:")

0: []
10: [-1.20627994  0.24768303 -0.90142902 -2.93856784 -2.8894218  -3.30769198
  0.51724106  0.60841632  1.50642312 -0.64717314]
20: [ 3.60338376 -3.39530361 -0.83634559  0.20642239 -0.99055593  0.53902053
  1.84907898  1.6751169   0.35512704  0.85250451  1.94873115 -2.58021793
  1.6327717   1.84535034  0.40279201 -0.35760159  0.32582158 -0.98578614
 -2.38491532 -2.0029312 ]
30: [ 0.265591   -0.82558218  4.05160807 -0.12876757 -1.95232683  3.03925634
  1.7813961  -0.83175136  2.34419626 -3.19270928 -1.33391369  0.7386521
  0.74374329 -0.99517421  0.54426881  1.20721646  0.37922703 -1.78257264
 -0.55362695 -1.65823189  0.51890907  0.82037895 -2.08953762 -4.6628544
 -1.40675433  0.67710588 -0.42832258  1.75799473 -0.17067132  3.94095685]
40: [-0.54340762  2.18029413  0.25043515 -0.50015432  4.42327187  0.10449424
 -0.7880773   3.69854449 -0.79028099 -0.39971043 -2.56643208  2.37252488
  0.78913334 -1.32527961  1.82872693  0.26896525 -2.64334855 -2.23605461
 -1.96874815 -2.82314601 -0

This takes about eight seconds, four seconds for the longest invocation invocation of `make_array`, `make_array(4)`  , and four seconds with longest invocation of `add_arrays`, when passed the results of `make_array(4)`. 

We did the right thing inside each list comprehension. We started the asynchronous tasks all at once and allowed Ray to handle the dependencies. Then we waited on one `ray.get()` call for all the output. 

However, what you see is no output and then everything is suddenly printed at once after eight seconds.

There are two fundamental problems with the way we've used `ray.get()` so far:

1. There's no timeout, in case something gets "hung".
2. We have to wait for _all_ the objects to be available before `ray.get()` returns.

The ability to specify a timeout is essential in production code as a defensive measure. Many potential problems could happen in a real production system, any one of which could cause the task we're waiting on to take an abnormally long time to complete or never complete. Our application would be deadlocked waiting on this task. Hence, it's **strongly recommended** in production software to always use timeouts on blocking calls, so that the application can attempt some sort of recovery in situations like this, or at least report the error and "degrade gracefully".

Actually, there _is_ a `timeout=<value>` option you can pass to `ray.get()` ([documentation](https://ray.readthedocs.io/en/latest/package-ref.html#ray.get)), but it will most likely be removed in a future release of Ray. Why remove it if timeouts are important? This change will simplify the implementation of `ray.get()` and encourage the use of `ray.wait()` for waiting ([documentation](https://ray.readthedocs.io/en/latest/package-ref.html#ray.wait)) instead, followed by using `ray.get()` to retrieve values for tasks that `ray.wait()` tells us are finished. 

Using `ray.wait()` is also the way to fix the second problem with using `ray.get()` by itself, that we have to wait for all tasks to finish before we get any values back. Some of those tasks finish more quickly in our contrived example. We would like to process those results as soon as they are available, even while others continue to run. We'll use `ray.wait()` for this purpose.

Therefore, while `ray.get()` is simple and convenient, for _production code_, we recommend using `ray.wait()`, **with** timeouts, for blocking on running tasks. Then use `ray.get()` to retrieve values of completed tasks. 

Here is the previous example rewritten to use `ray.wait()`:

In [7]:
start = time.time()
array_refs = [make_array.remote(n*10) for n in range(5)]
added_array_refs = [add_arrays.remote(ref, ref) for ref in array_refs]

arrays = []
waiting_refs = list(added_array_refs)  # Assign a working list to the full list of refs
while len(waiting_refs) > 0:           # Loop until all tasks have completed
    # Call ray.wait with:
    #   1. the list of refs we're still waiting to complete,
    #   2. tell it to return immediately as soon as one of them completes,
    #   3. tell it wait up to 10 seconds before timing out.
    ready_refs, remaining_refs = ray.wait(waiting_refs, num_returns=1, timeout=10.0)
    print('Returned {:3d} completed tasks. (elapsed time: {:6.3f})'.format(len(ready_refs), time.time() - start))
    new_arrays = ray.get(ready_refs)
    arrays.extend(new_arrays)
    for array in new_arrays:
        print(f'{array.size}: {array}')
    waiting_refs = remaining_refs  # Reset this list; don't include the completed refs in the list again!
    
print(f"\nall arrays: {arrays}")
pd(time.time() - start, prefix="Total time:")

Returned   1 completed tasks. (elapsed time:  0.078)
0: []
Returned   1 completed tasks. (elapsed time:  2.040)
10: [ 0.45437406  0.69331209 -2.58309259  2.76158254  3.73382934 -3.31166676
  0.36643433 -0.53072721 -4.28763215  1.13132442]
Returned   1 completed tasks. (elapsed time:  4.060)
20: [ 0.79948334  0.70030982  2.13493499  0.77234785  0.1529277  -0.19818794
  0.55499732 -1.24079467  0.78218259  2.871603   -0.57683763 -0.08803753
  0.20341453 -0.52553137 -0.46152076  1.61664059  0.40143076  1.58148853
  2.61301829  0.25961314]
Returned   1 completed tasks. (elapsed time:  6.081)
30: [-0.35357419 -1.29585987 -0.66793575  1.51099812 -0.58083462  0.99669201
  1.2752888  -0.37636809 -3.01727583 -2.65229258  3.09931129 -1.89746882
 -0.48778671  0.35430921  3.1670273  -1.11434611  0.92733488 -0.63265432
 -0.65913595 -1.50830651  1.82529042 -0.07771442 -0.11920146  0.70151131
  1.01352384  2.5904357   3.45541048  0.75763957  2.52362022  1.05556294]
Returned   1 completed tasks. (elaps

Now it still takes about 8 seconds to complete, 4 seconds for the longest invocation of `make_array` and 4 seconds for the invocation of `add_arrays`, but since the others complete more quickly, we see their results as soon as they become available, at 0, 2, 4, and 6 second intervals.

> **Warning:** For each call to `ray.wait()` in a loop like this, it's important to remove the refs that have completed. Otherwise, `ray.wait()` will return immediately with the same list containg the first completed item, over and over again; you'll loop forever!! Resetting the list is easy, since the second list returned by `ray.wait()` is the rest of the items that are still running. So, that's what we use.

Now let's try it with `num_returns = 2`:

In [8]:
start = time.time()
array_refs = [make_array.remote(n*10) for n in range(5)]
added_array_refs = [add_arrays.remote(ref, ref) for ref in array_refs]

arrays = []
waiting_refs = list(added_array_refs)  # Assign a working list to the full list of refs
while len(waiting_refs) > 0:           # Loop until all tasks have completed
    # Call ray.wait with:
    #   1. the list of refs we're still waiting to complete,
    #   2. tell it to return immediately as soon as TWO of them complete,
    #   3. tell it wait up to 10 seconds before timing out.
    return_n = 2 if len(waiting_refs) > 1 else 1
    ready_refs, remaining_refs = ray.wait(waiting_refs, num_returns=return_n, timeout=10.0)
    print('Returned {:3d} completed tasks. (elapsed time: {:6.3f})'.format(len(ready_refs), time.time() - start))
    new_arrays = ray.get(ready_refs)
    arrays.extend(new_arrays)
    for array in new_arrays:
        print(f'{array.size}: {array}')
    waiting_refs = remaining_refs  # Reset this list; don't include the completed refs in the list again!
    
print(f"\nall arrays: {arrays}")
pd(time.time() - start, prefix="Total time:")

Returned   2 completed tasks. (elapsed time:  2.013)
0: []
10: [-2.94487463 -0.20515211 -1.21721959 -2.54505005  2.53539694 -2.89842961
 -2.32593868  0.4861679  -2.57712589 -1.74041328]
Returned   2 completed tasks. (elapsed time:  6.014)
20: [ 0.15756339  0.70655518  0.43356913  0.64729627 -0.78901818 -2.0719179
  0.9722761   0.4325182  -2.26739337 -0.08784976  0.17736519 -3.39512875
  1.5598177  -0.83083243 -3.92696585 -2.60103557 -2.56880192  0.5517638
  0.99855315 -2.16416783]
30: [-0.07768504 -0.70268598  0.92056113 -1.71179513 -1.318743   -5.69315124
 -0.037428   -0.85751717  3.70692497 -1.67805509  0.0671539   2.6207001
  1.23987118  0.92102111  1.77791423  2.30275174  2.73170802 -2.82441125
 -1.81794507 -2.28215284  2.05025238 -2.46189908 -0.11176304 -1.16338214
 -2.80320196  0.91222451 -2.45406968 -0.97807619 -2.76382338  1.9662463 ]
Returned   1 completed tasks. (elapsed time:  8.011)
40: [ 1.176072    1.19516666 -1.41519632  2.40864955  0.26778945  0.69992192
  1.4436109   0

Now we get two at a time output. Note that we don't actually pass `num_returns=2` every time. If you ask for more items than the length of the input list, you get an error. So, we compute `num_returns`, using `2` except when there's only one task to wait on, in which case we use `1`. So, in fact, the output for `40` was a single task result, because we started with `5` and processed two at a time.

 For a longer discussion on `ray.wait()`, see [this blog post](https://medium.com/distributed-computing-with-ray/ray-tips-and-tricks-part-i-ray-wait-9ed7a0b9836d).

## Exercise 1

The following cell is identical to the last one. Modify it to use a timeout of `2.5` seconds, shorter than our longest tasks. What happens now? Try using other times.

See the [solutions notebook](solutions/Advanced-Ray-Solutions.ipynb) for a discussion of this exercise and the subsequent exercises.

In [8]:
start = time.time()
array_refs = [make_array.remote(n*10) for n in range(5)]
added_array_refs = [add_arrays.remote(ref, ref) for ref in array_refs]

arrays = []
waiting_refs = list(added_array_refs)  # Assign a working list to the full list of refs
while len(waiting_refs) > 0:           # Loop until all tasks have completed
    # Call ray.wait with:
    #   1. the list of refs we're still waiting to complete,
    #   2. tell it to return immediately as soon as TWO of them complete,
    #   3. tell it wait up to 10 seconds before timing out.
    return_n = 2 if len(waiting_refs) > 1 else 1
    ready_refs, remaining_refs = ray.wait(waiting_refs, num_returns=return_n, timeout=2.0)
    print('Returned {:3d} completed tasks. (elapsed time: {:6.3f})'.format(len(ready_refs), time.time() - start))
    new_arrays = ray.get(ready_refs)
    arrays.extend(new_arrays)
    for array in new_arrays:
        print(f'{array.size}: {array}')
    waiting_refs = remaining_refs  # Reset this list; don't include the completed refs in the list again!
    
print(f"\nall arrays: {arrays}")
pd(time.time() - start, prefix="Total time:")

Returned   1 completed tasks. (elapsed time:  2.025)
0: []
Returned   1 completed tasks. (elapsed time:  4.053)
10: [-0.97231068  2.52315036  4.35752675 -0.92818539 -0.77902069  2.67368305
  1.97182728  1.27748676 -0.99547691  1.17135048]
Returned   2 completed tasks. (elapsed time:  6.078)
20: [ 2.94098759  0.97851975 -0.35555633 -2.85215394 -1.98540851  0.76162399
  0.65003658  0.02221309 -2.67200912 -0.81689789 -0.24697521 -2.84730499
 -2.55390456 -0.45914041  2.58095066 -0.21837078 -3.99672475  2.48883392
 -0.57055715  1.23068673]
30: [ 2.5367951   1.026691    0.61389146 -1.15582955  0.06334166  0.6462227
  0.05842671  2.20924391  1.78558802  1.56697406 -5.20857077  1.79145787
  1.09864293  2.61096621 -0.70187498 -3.39236698 -0.42760105 -3.3835307
 -0.3326951  -1.3224836   0.19472809 -0.94375977 -1.85014524 -0.23699481
 -0.98814579 -2.57689544 -0.63241467 -2.31054493  1.60880676 -1.72148001]
Returned   1 completed tasks. (elapsed time:  8.070)
40: [ 2.13079848 -0.3580135   0.698734

In conclusion:

> **Tips:**
>
> 1. Use `ray.wait()` with a timeout to wait for one or more running tasks. Then use `ray.get()` to retrieve the values for the finished tasks.
> 2. When looping over calls to `ray.wait()` with a list of object refs for running tasks, remove the previously-completed and retrieved objects from the list.
> 3. Don't ask for results you don't need.
> 4. Don't ask for the results you need until you really need them.

## Exercise 3

Let's practice converting a slow loop to Ray, including using `ray.wait()`. Change the function to be a Ray task. Change the invocations to use the `ray.wait()` idiom. You can just use the default values for `num_returns` and `timeout` if you want. The second cell uses `assert` statements to check your work.

In [11]:
@ray.remote
def slow_square(n):
    time.sleep(n)
    return n*n

start = time.time()
squares = [slow_square.remote(n) for n in range(4)]
remaining_refs = list(squares)
squares = []
while len(remaining_refs) > 0:
    ready_refs, remaining_refs = ray.wait(remaining_refs, num_returns=1, timeout=10.0)
    for ref in ready_refs:
        square = ray.get(ref)
        squares.append(square)
    
# for square in squares:
#     print (f'finished: {square}')
duration = time.time() - start

In [12]:
assert squares == [0, 1, 4, 9], f'Did you use ray.get() to retrieve the values? squares = {squares}'
assert duration < 4.1, f'Did you use Ray to parallelize the work? duration = {duration}' 

## Limiting Task Invocations and Retries on Failure

> **Note:** This feature may change in a future version of Ray. See the latest details in the [Ray documentation](https://docs.ray.io/en/latest/package-ref.html#ray.remote). 

Two options you can pass to `ray.remote` when defining a task affect how often it can be invoked and retrying on failure:

* `max_calls`: This specifies the maximum number of times that a given worker can execute the given remote function before it must exit. This can be used to address memory leaks in third-party libraries or to reclaim resources that cannot easily be released, e.g., GPU memory that was acquired by TensorFlow. By default this is infinite.
* `max_retries`: This specifies the maximum number of times that the remote function should be rerun when the worker process executing it crashes unexpectedly. The minimum valid value is 0, the default is 4, and a value of -1 indicates infinite retries are allowed.

Example:

```python
@ray.remote(max_calls=10000, max_retries=10)
def foo():
    pass
```

See the [ray.remote()](https://docs.ray.io/en/latest/package-ref.html#ray.remote) documentation for all the keyword arguments supported.

### Overriding with config()

Remote task and actor objects returned by `@ray.remote` can also be dynamically modified with the same arguments supported by `ray.remote()` using `options()` as in the following examples:

```python
@ray.remote(num_gpus=1, max_calls=1, num_return_vals=2)
def f():
    return 1, 2
g = f.options(num_gpus=2, max_calls=None)
```

## What Is the Optimal Task Granularity

How fine-grained should Ray tasks be? There's no fixed rule of thumb, but Ray clearly adds some overhead for task management and using object stores in a cluster. Therefore, it makes sense that tasks which are too small will perform poorly.

We'll explore this topic over several more lessons, but for now, let's get a sense of the overhead while running in your setup.

We'll continue to use NumPy arrays to create "load", but remove the `sleep` calls:

In [13]:
def noop(n):
    return n

def local_make_array(n):
    return np.random.standard_normal(n)

@ray.remote
def remote_make_array(n):
    return local_make_array(n)

Let's do `trials` runs for each experiment, to average out background noise:

In [14]:
trials=100

First, let's use `noop` to baseline local function calls. Note that we call `print` for the duration, rathern than `pd`, because the overhead is so low the `pd` formatting will print `0.000`:

In [15]:
start = time.time()
[noop(t) for t in range(trials)]
print(f'{time.time() - start} seconds')

0.00013399124145507812 seconds


Let's try the same run with `local_make_array(n)` for `n = 100000`:

In [16]:
start = time.time()
[local_make_array(100000) for _ in range(trials)]
print(f'{time.time() - start} seconds')

0.32164716720581055 seconds


So, we can safely ignore the "noop" overhead for now. For completeness, here's what happens with remote execution:

In [17]:
start = time.time()
refs = [remote_make_array.remote(100000) for _ in range(trials)]
ray.get(refs)
print(f'{time.time() - start} seconds')

0.8308298587799072 seconds


For arrays of 100000, using Ray is faster (at least on this test machine). The benefits of parallel computation, rather than synchronous, already outweight the Ray overhead.

## Exercise 4

1. Try incrementing size of n to 2n
2. Do you see a marked difference between the local vs remote times?
3. For the brave, try using `mathplotlib` to plot it.

## Profiling Tasks with ray.timeline()

Sometimes you need to debug performance problems in Ray tasks. Calling `ray.timeline(file)` ([documentation](https://ray.readthedocs.io/en/latest/package-ref.html#ray.timeline)) captures profiling information for subsequent task execution to the specified file. Afterwards, you can view the data in a Chrome web browser. The format used is unique to Chrome, so Chrome has be used to view the data.

Let's try it with our `make_array` and `add_arrays` methods in the following code. First some potential cleanup:

In [18]:
timeline_file = 'task-timeline.txt' # Will be found in the same directory as this notebook.
if os.path.isfile(timeline_file):   # Delete old one, if an old one exists already.
    os.remove(timeline_file)

In [19]:
ray.timeline(timeline_file)
start = time.time()
array_refs = [make_array.remote(n*10) for n in range(5)]
added_array_refs = [add_arrays.remote(ref, ref) for ref in array_refs]
for array in ray.get(added_array_refs):
    print(f'{array.size}: {array}')
pd(time.time() - start, prefix="Total time:")

0: []
10: [-0.42872456  2.39275388  2.40493511  2.25528772  1.80162009 -0.66377265
  4.47916717  1.32659197 -1.25103939  0.48951071]
20: [-1.59360357 -1.19432478  1.05589118 -2.16837916 -0.41610622 -1.87279165
  2.26415468 -0.38468836  0.24328454 -1.10790923  2.18465994 -0.16410272
 -3.97105582 -1.05111702 -2.67922779 -1.29224434 -1.95085881  2.98431342
 -1.05068339  2.34006329]
30: [ 1.09087786e+00  5.02467669e-01  2.37519655e+00  5.62429028e-01
 -1.11793526e+00  2.71136693e+00  2.20488368e-01  6.58302434e-02
  2.42449403e+00  1.90956966e+00  1.69420400e+00 -2.01783201e+00
  2.98122236e-01 -9.50686402e-01 -7.12463774e-02  2.28202281e-03
  3.20246618e-01  1.75816279e+00 -2.08617635e-02 -2.13128869e+00
 -2.65032416e-01  3.12530684e+00 -3.13685872e+00 -1.74585437e+00
  1.59550289e+00  2.62103030e+00 -2.05301113e+00  1.42247003e+00
  1.41060169e+00  1.56025987e+00]
40: [-1.12539303 -2.68176262 -0.02710309 -1.56164667  0.15347826  0.18451274
 -1.39272434 -1.40963782 -0.54166624 -0.16024986

Now, to view the data:

1. Open Chrome and enter chrome://tracing.
2. Click the _Load_ button to load the `task-timeline.txt` file, which will be in this notebook's directory. 
3. To zoom in or out, click the "asymmetric" up-down arrow button. Then hold the mouse button in the graph and roll the mouse scroll wheel up or down. (On a laptop trackpad, press and hold, then use another finger to slide up and down.)
4. To move around, click the crossed arrow and drag a section in view. 
5. Click on a box in the timeline to see details about it. 

Look for blocks corresponding to long-running tasks and look for idle periods, which reflect processing outside the context of Ray.

Here is a screen grab profiling the previous code, zoomed in on one block of tasks and with one task selected. Note the processes shown on the left for drivers (more than one notebook was running at this time) and workers.

![Ray Trace Example](../images/Ray-Trace-Example.png)

In [20]:
ray.shutdown()  # "Undo ray.init()".

The next lesson, [Ray Actors Revisited](02-Ray-Actors-Revisited.ipynb), revisits actors. It provides a more in-depth look at actor characteristics and profiling actor performance using the _Ray Dashboard_.